# Task 6: Bidirectional LSTM (BiLSTM) from Scratch

**Goal:** Implement the main LSTM gate equations using PyTorch tensor operations and NumPy.

In [1]:
import torch
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


## 1. Simple Padded Sequence Data

Here we use two sequences with different lengths. Zero padding is used for the shorter sequence.

In [2]:
# Shape: batch, time steps, features
X = torch.tensor([
    [[1.0, 0.5], [0.8, 0.2], [0.4, 0.1], [0.0, 0.0]],
    [[0.2, 0.9], [0.3, 0.7], [0.6, 0.4], [0.5, 0.2]]
])

# Actual sequence lengths before padding
lengths = [3, 4]

print("Input shape:", X.shape)
print("Sequence lengths:", lengths)

Input shape: torch.Size([2, 4, 2])
Sequence lengths: [3, 4]


## 2. LSTM Parameters

The LSTM uses four gates:

- Input gate `i`
- Forget gate `f`
- Output gate `o`
- Cell candidate `g`

In [3]:
input_size = 2
hidden_size = 3

# One set of parameters for each direction
def create_parameters():
    W = torch.randn(4 * hidden_size, input_size + hidden_size) * 0.2
    b = torch.zeros(4 * hidden_size)
    return W, b

W_forward, b_forward = create_parameters()
W_backward, b_backward = create_parameters()

## 3. LSTM Step Function

In [4]:
def lstm_step(x, h, c, W, b):
    # Combine current input and previous hidden state.
    combined = torch.cat([x, h], dim=1)

    # Calculate all four gate values together.
    gates = combined @ W.T + b

    i, f, o, g = torch.chunk(gates, 4, dim=1)

    # Gate activations
    i = torch.sigmoid(i)
    f = torch.sigmoid(f)
    o = torch.sigmoid(o)
    g = torch.tanh(g)

    # Update cell state
    c_new = f * c + i * g

    # Update hidden state
    h_new = o * torch.tanh(c_new)

    return h_new, c_new

## 4. Forward LSTM

The sequence is processed from the first time step to the last time step.

In [5]:
def forward_lstm(X, lengths, W, b):
    batch_size, time_steps, _ = X.shape

    h = torch.zeros(batch_size, hidden_size)
    c = torch.zeros(batch_size, hidden_size)

    outputs = []

    for t in range(time_steps):

        h_new, c_new = lstm_step(
            X[:, t, :], h, c, W, b
        )

        # Keep the previous state for padded positions.
        mask = (
            torch.tensor(
                [t < length for length in lengths]
            ).float().unsqueeze(1)
        )

        h = mask * h_new + (1 - mask) * h
        c = mask * c_new + (1 - mask) * c

        outputs.append(h)

    return torch.stack(outputs, dim=1)

## 5. Backward LSTM

The same LSTM equations are used, but the sequence is processed from the last valid time step toward the first.

This is the backward direction of BiLSTM.

In [6]:
def backward_lstm(X, lengths, W, b):
    batch_size, time_steps, _ = X.shape

    h = torch.zeros(batch_size, hidden_size)
    c = torch.zeros(batch_size, hidden_size)

    outputs = [None] * time_steps

    for t in range(time_steps - 1, -1, -1):

        h_new, c_new = lstm_step(
            X[:, t, :], h, c, W, b
        )

        mask = (
            torch.tensor(
                [t < length for length in lengths]
            ).float().unsqueeze(1)
        )

        h = mask * h_new + (1 - mask) * h
        c = mask * c_new + (1 - mask) * c

        outputs[t] = h

    return torch.stack(outputs, dim=1)

## 6. Bidirectional LSTM

The forward and backward outputs are concatenated.

For each time step:

`BiLSTM output = [forward output ; backward output]`

In [7]:
forward_output = forward_lstm(
    X, lengths, W_forward, b_forward
)

backward_output = backward_lstm(
    X, lengths, W_backward, b_backward
)

# Join the two directions
bilstm_output = torch.cat(
    [forward_output, backward_output],
    dim=2
)

print("Forward output shape:", forward_output.shape)
print("Backward output shape:", backward_output.shape)
print("BiLSTM output shape:", bilstm_output.shape)

Forward output shape: torch.Size([2, 4, 3])
Backward output shape: torch.Size([2, 4, 3])
BiLSTM output shape: torch.Size([2, 4, 6])


## 7. Display the Final Output

The output size is `hidden_size × 2` because we combine the forward and backward hidden states.

In [8]:
print("BiLSTM output:")
print(bilstm_output)

print("\nFinal valid output of sequence 1:")
print(bilstm_output[0, lengths[0] - 1])

print("\nFinal output of sequence 2:")
print(bilstm_output[1, lengths[1] - 1])

BiLSTM output:
tensor([[[ 0.0549,  0.1174, -0.0338, -0.0091, -0.0845, -0.0443],
         [ 0.0588,  0.1496, -0.0444, -0.0046, -0.0645, -0.0284],
         [ 0.0491,  0.1244, -0.0386, -0.0009, -0.0282, -0.0128],
         [ 0.0491,  0.1244, -0.0386,  0.0000,  0.0000,  0.0000]],

        [[ 0.0782,  0.0466, -0.0142, -0.0036, -0.0332, -0.0436],
         [ 0.1035,  0.0818, -0.0214, -0.0047, -0.0428, -0.0367],
         [ 0.0957,  0.1242, -0.0313, -0.0036, -0.0532, -0.0307],
         [ 0.0756,  0.1283, -0.0332, -0.0008, -0.0345, -0.0175]]])

Final valid output of sequence 1:
tensor([ 0.0491,  0.1244, -0.0386, -0.0009, -0.0282, -0.0128])

Final output of sequence 2:
tensor([ 0.0756,  0.1283, -0.0332, -0.0008, -0.0345, -0.0175])
